# Level 6: Evaluation & Validation

You've fine-tuned a model. But is it actually better? How much better? Answering these questions is the job of **evaluation**.

Evaluating generative AI is notoriously difficult because there's often no single "correct" answer. A good response can be phrased in many ways. This is different from traditional machine learning tasks like classification, where you can simply check if the predicted label is correct.

This notebook covers the main approaches to evaluating your fine-tuned LLM.

### Step 1: Install Dependencies

We'll use the Hugging Face `evaluate` library, which provides easy access to many common metrics.

In [ ]:
!pip install -q evaluate transformers torch

### 6.1 Automatic Metrics

These metrics compare the model's generated output against a ground-truth reference text. They are fast and cheap but often limited.

#### Perplexity (PPL)

**What it is**: A measure of how well a probability model predicts a sample. In LLM terms, it's a measure of the model's "surprise" when encountering a piece of text. **Lower perplexity is better**.

**When to use it**: It's most useful for evaluating the fluency of a model after continued pre-training, not for evaluating instruction-following ability.

In [ ]:
import torch
import evaluate
from transformers import AutoModelForCausalLM

perplexity = evaluate.load("perplexity", module_type="metric")
model_id = "distilgpt2" # Using a small model for this demo

data = ["This is a test sentence.", "The quick brown fox jumps over the lazy dog."]

results = perplexity.compute(model_id=model_id, add_start_token=False, data=data)
print(f"Perplexity: {results['mean_perplexity']:.2f}")

#### BLEU & ROUGE

- **BLEU (Bilingual Evaluation Understudy)**: Measures precision - how much of the generated text appears in the reference. It was originally designed for machine translation.
- **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**: Measures recall - how much of the reference text appears in the generated output. It's good for summarization tasks.

**Limitation**: Both are based on n-gram (sequence of words) overlap and don't understand semantics. "The cat sat on the mat" and "On the mat, the cat sat" have different scores but mean the same thing.

In [ ]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

prediction = ["the cat sat on the mat"]
reference = [["the cat is on the mat"]]

bleu_score = bleu.compute(predictions=prediction, references=reference)
rouge_score = rouge.compute(predictions=prediction, references=reference)

print(f"BLEU Score: {bleu_score['bleu']:.2f}")
print(f"ROUGE-L Score: {rouge_score['rougeL']:.2f}")

#### BERTScore

**What it is**: A more advanced metric that computes the semantic similarity between a prediction and a reference by embedding them with another model (like BERT) and comparing the cosine similarity. It captures meaning much better than BLEU or ROUGE.

**Higher BERTScore is better**.

In [ ]:
bertscore = evaluate.load("bertscore")

prediction = ["The cat sat on the mat"]
reference = [["A feline was resting on the rug"]]

results = bertscore.compute(predictions=prediction, references=reference, lang="en")
# We look at the F1 score, which is a combination of precision and recall
print(f"BERTScore F1: {torch.mean(results['f1']).item():.2f}")

### 6.2 LLM-as-a-Judge

A powerful, emerging technique is to use a very capable LLM (like GPT-4 or Claude 3 Opus) as an impartial "judge" to evaluate the quality of your fine-tuned model's output.

**How it works**:
1. You give the judge LLM the original prompt.
2. You provide the response from your fine-tuned model.
3. You ask the judge to score the response based on a set of criteria (e.g., helpfulness, correctness, clarity) and to provide a rationale.

This is highly scalable and correlates well with human judgment.

#### Conceptual Example of a Judge Prompt

In [ ]:
judge_prompt = """
You are an impartial AI assistant expert tasked with evaluating the quality of a model's response.

You will be given a user's prompt and a model's response. Your task is to score the response on a scale of 1 to 10 for 'helpfulness'.

Provide your evaluation in a JSON format with two keys: 'score' and 'rationale'.

--- PROMPT ---
Explain the concept of quantum entanglement in simple terms.

--- RESPONSE ---
Quantum entanglement is a physical phenomenon that occurs when a pair or group of particles is generated in such a way that the quantum state of each particle of the pair or group cannot be described independently of the state of the others, even when the particles are separated by a large distance.

--- EVALUATION ---
"""
print(judge_prompt)
# You would then send this prompt to a powerful LLM API (like OpenAI's or Anthropic's) to get the evaluation.

### 6.3 Human Evaluation

**This is the gold standard.**

Ultimately, the best way to know if a model is performing well is to have humans evaluate its outputs. Common methods include:

- **A/B Testing**: Show users the output from two different models (e.g., the base model vs. your fine-tuned model) and ask which one they prefer.
- **Likert Scales**: Ask evaluators to rate responses on a scale (e.g., 1-5) for specific attributes like correctness, helpfulness, and tone.